### Step 1: Install necesscary packages

In [12]:
!pip install matplotlib
!pip install torch numpy transformers datasets tiktoken wandb tqdm

86790.07s - pydevd: Sending message related to process being replaced timed-out after 5 seconds



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


86796.01s - pydevd: Sending message related to process being replaced timed-out after 5 seconds



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


### Step 2: Package imports and configuration

In [13]:
import sys
import os
sys.path.append(os.path.abspath("..")) 
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pickle
from model import GPT, GPTConfig
import random
from tqdm import tqdm
import time
import json
import matplotlib.pyplot as plt
# Configuration
beta = 0.5
device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_lr = 1e-4
epochs = 5
batch_size = 4 #64
max_length =64
num_samples = 1
max_new_tokens = 200
temperature = 0.8
top_k = 200
# tokenizer
with open("../sft/meta.pkl", "rb") as f:
    meta = pickle.load(f)
stoi, itos = meta["stoi"], meta["itos"]
def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])

### Step 3: Define helper functions

In [14]:
def compute_logprob(input_ids):
    inputs = input_ids[:, :-1]
    targets = input_ids[:, 1:]
    logits, _ = gpt(inputs, full_seq=True)
    B, T, V = logits.size()
    logits_flat = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)
    loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=0, reduction='none')
    loss = loss.reshape(B, T)
    attention_mask = (targets != 0).float()
    loss = (loss * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)
    return -loss 

def pad_or_truncate(seq, max_length):
    return seq[-max_length:] if len(seq) > max_length else seq + [0] * (max_length - len(seq))

def get_batches(lines, batch_size):
    random.shuffle(lines)
    #for l in lines:
    #    print(l[1])
    for i in range(0, len(lines), batch_size):
        batch = lines[i:i+batch_size]
        if len(batch) < batch_size:
            continue
        neg_inputs = [pad_or_truncate(encode(p['negative'] + '\n\n\n\n'), max_length) for p in batch]
        pos_inputs = [pad_or_truncate(encode(p['positive'] + '\n\n\n\n'), max_length) for p in batch]
        neg_tensor = torch.tensor(neg_inputs, dtype=torch.long, device=device)
        pos_tensor = torch.tensor(pos_inputs, dtype=torch.long, device=device)
        yield neg_tensor, pos_tensor

### Step 4: Load the pretrained NanoGPT model

In [15]:
ckpt = torch.load("../sft/gpt.pt", map_location=device)
gptconf = GPTConfig(**ckpt['model_args'])
gpt = GPT(gptconf)
state_dict = ckpt['model']
unwanted_prefix = '_orig_mod.'
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
gpt.to(device).train()

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(74, 348)
    (wpe): Embedding(256, 348)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=348, out_features=1044, bias=False)
          (c_proj): Linear(in_features=348, out_features=348, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=348, out_features=1392, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1392, out_features=348, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=348, out_features=74, bias=False)
)

### Generating Dataset

In [16]:

import random
import json

random.seed(42)

data = []
seen_prompts = set()

# Define the 12 equation types
equation_types = [
    'addition', 'subtraction', 'multiplication', 'division',
    'addition_unknown', 'addition2_unknown',
    'subtraction_unknown', 'subtraction2_unknown',
    'multiplication_unknown', 'multiplication2_unknown',
    'division_unknown', 'division2_unknown'
]

# Allocate samples across arithmetic and algebra equations
samples_per_type = {
    'addition': 17000,
    'subtraction': 12000,
    'multiplication': 3500,
    'division': 4000,
    'addition_unknown': 14000,
    'addition2_unknown': 14000,
    'subtraction_unknown': 14000,
    'subtraction2_unknown': 12000,
    'multiplication_unknown': 2500,
    'multiplication2_unknown': 2500,
    'division_unknown': 2500,
    'division2_unknown': 2500
}

print("Total target samples:", sum(samples_per_type.values()))
print("Generating positive-negative pairs...\n")

# Generate samples for each equation type
for eq_type in equation_types:
    generated = 0
    attempts = 0
    target_samples = samples_per_type[eq_type]
    max_attempts = target_samples * 20

    while generated < target_samples and attempts < max_attempts:
        attempts += 1

        if eq_type == 'addition':
            a = random.randint(0, 199)
            b = random.randint(0, 199)
            prompt = f"{a}+{b}=?"
            answer = a + b
            positive = f"The answer is {answer} because {a}+{b} equals {answer}."

        elif eq_type == 'subtraction':
            a = random.randint(0, 199)
            b = random.randint(0, 199)

            if b > a:
                a, b = b, a

            prompt = f"{a}-{b}=?"
            answer = a - b
            positive = f"The answer is {answer} because {a}-{b} equals {answer}."

        elif eq_type == 'multiplication':
            a = random.randint(1, 20)
            b = random.randint(0, 200)
            prompt = f"{a}*{b}=?"
            answer = a * b
            positive = f"The answer is {answer} because {a}*{b} equals {answer}."

        elif eq_type == 'division':
            b = random.randint(1, 30)
            answer = random.randint(0, 200)
            a = b * answer
            prompt = f"{a}/{b}=?"
            positive = f"The answer is {answer} because {a}/{b} equals {answer}."

        elif eq_type == 'addition_unknown':
            answer = random.randint(0, 199)
            a = random.randint(0, 199)
            b = a + answer
            prompt = f"{a}+x={b},x=?"
            positive = f"The answer is {answer} because {b}-{a} equals {answer}."

        elif eq_type == 'addition2_unknown':
            a = random.randint(0, 199)
            answer = random.randint(0, 199)
            b = answer + a
            prompt = f"x+{a}={b},x=?"
            positive = f"The answer is {answer} because {b}-{a} equals {answer}."

        elif eq_type == 'subtraction_unknown':
            answer = random.randint(0, 199)
            b = random.randint(0, 199)
            a = answer + b
            prompt = f"{a}-x={b},x=?"
            positive = f"The answer is {answer} because {a}-{b} equals {answer}."

        elif eq_type == 'subtraction2_unknown':
            answer = random.randint(0, 199)
            a = random.randint(0, answer)
            b = answer - a
            prompt = f"x-{a}={b},x=?"
            positive = f"The answer is {answer} because {b}+{a} equals {answer}."

        elif eq_type == 'multiplication_unknown':
            a = random.randint(1, 20)
            answer = random.randint(0, 200)
            b = a * answer
            prompt = f"{a}*x={b},x=?"
            positive = f"The answer is {answer} because {b}/{a} equals {answer}."

        elif eq_type == 'multiplication2_unknown':
            a = random.randint(1, 20)
            answer = random.randint(0, 200)
            b = a * answer
            prompt = f"x*{a}={b},x=?"
            positive = f"The answer is {answer} because {b}/{a} equals {answer}."

        elif eq_type == 'division_unknown':
            answer = random.randint(1, 20)
            b = random.randint(1, 200)
            a = answer * b
            prompt = f"{a}/x={b},x=?"
            positive = f"The answer is {answer} because {a}/{b} equals {answer}."

        elif eq_type == 'division2_unknown':
            a = random.randint(1, 20)
            b = random.randint(0, 200)
            answer = b * a
            prompt = f"x/{a}={b},x=?"
            positive = f"The answer is {answer} because {a}*{b} equals {answer}."

        # Skip duplicate questions
        if prompt in seen_prompts:
            continue

        seen_prompts.add(prompt)

        negative = "Sorry, I do not know."

        data.append({
            "negative": prompt + " " + negative,
            "positive": prompt + " " + positive
        })

        generated += 1

    # Warn if target is not reached
    if generated < target_samples:
        print(
            f"WARNING: {eq_type} came up short: "
            f"{generated}/{target_samples}"
        )

    print(f"✓ {eq_type}: {generated} samples generated")

print(f"\n✓ Total generated: {len(data)} unique pairs")

# Shuffle samples to mix equation types
random.shuffle(data)

# Save to JSON
with open("pos_neg_pairs.json", "w") as f:
    json.dump(data, f)

print("✓ Saved to pos_neg_pairs.json")

# Validate dataset size and uniqueness
assert len(data) == sum(samples_per_type.values()), \
    "Dataset size does not match the target."

assert len(seen_prompts) == len(data), \
    "Duplicate prompts detected."

print("✓ Dataset size and uniqueness validated")


Total target samples: 100500
Generating positive-negative pairs...

✓ addition: 17000 samples generated
✓ subtraction: 12000 samples generated
✓ multiplication: 3500 samples generated
✓ division: 4000 samples generated
✓ addition_unknown: 14000 samples generated
✓ addition2_unknown: 14000 samples generated
✓ subtraction_unknown: 14000 samples generated
✓ subtraction2_unknown: 12000 samples generated
✓ multiplication_unknown: 2500 samples generated
✓ multiplication2_unknown: 2500 samples generated
✓ division_unknown: 2500 samples generated
✓ division2_unknown: 2500 samples generated

✓ Total generated: 100500 unique pairs
✓ Saved to pos_neg_pairs.json
✓ Dataset size and uniqueness validated


### Step 5: Load Data (**students are required to complete this part!**)

A possible issue: stoi.keys does not include '!', thus '!' cannot be used in the data pairs of pos_neg_pairs.json.

In [17]:
print(stoi.keys())

dict_keys(['\n', ' ', "'", '*', '+', ',', '-', '.', '/', '=', '?', '’', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z'])


In [18]:

with open("pos_neg_pairs.json") as f:
    lines = json.load(f)
    #### check our data here
    print(type(lines))
    print("Total number of pairs:", len(lines))
    for item in random.sample(lines,5): 
        print(item['negative'])
        print(item['positive'])
        print("-----")

#### clean data just in case
for pair in lines:
    for key in ['negative', 'positive']:
        pair[key] = "".join(char for char in pair[key] if char in stoi)

<class 'list'>
Total number of pairs: 100500
259-x=76,x=? Sorry, I do not know.
259-x=76,x=? The answer is 183 because 259-76 equals 183.
-----
199-98=? Sorry, I do not know.
199-98=? The answer is 101 because 199-98 equals 101.
-----
177-68=? Sorry, I do not know.
177-68=? The answer is 109 because 177-68 equals 109.
-----
x*9=702,x=? Sorry, I do not know.
x*9=702,x=? The answer is 78 because 702/9 equals 78.
-----
199+60=? Sorry, I do not know.
199+60=? The answer is 259 because 199+60 equals 259.
-----


### Step 6: Build the optimizer and scheduler (**students are required to complete this part!**)

In [8]:
# recommend to use the AdamW optimizer 
optimiser = torch.optim.AdamW(gpt.parameters(), lr=base_lr)
scheduler = torch.optim.lr_scheduler.StepLR(optimiser, step_size=2, gamma=0.5)

### Step 7: Begin training (**students are required to complete this part!**)

In [10]:
total_steps = len(lines) // batch_size
#### 
# to monitor loss
epoch_loss = []
####
for epoch in range(epochs):
    ####
    # to monitor loss
    loss_history = []
    ####
    pbar = tqdm(get_batches(lines, batch_size))
    for step, (neg_tensor,pos_tensor) in enumerate(pbar):
        ###########################################################
        # Please complete the training code here!
        # Examples: 
        # ...
        # neg_logprob
        # pos_logprob 
        # loss = -F.logsigmoid((pos_logprob - neg_logprob) / beta).mean() - pos_logprob.mean() * 0.1 
        # ...
        
        optimiser.zero_grad()

        neg_logprob = compute_logprob(neg_tensor)
        pos_logprob = compute_logprob(pos_tensor)

        loss = -F.logsigmoid((pos_logprob - neg_logprob) / beta).mean() - pos_logprob.mean() * 0.1 
        loss.backward()

        optimiser.step()

        loss_val = loss.item()
        loss_history.append(loss_val)

    avg_loss = sum(loss_history) / len(loss_history)
    epoch_loss.append(avg_loss)
    scheduler.step()
        ###########################################################
    ckpt_path = f"./dpo.pt"
    torch.save({
        "model_state_dict": gpt.state_dict(),
        "model_args": ckpt['model_args'],
    }, ckpt_path)
    print(f"Saved checkpoint to {ckpt_path}")

#### here we print the loss...
epoch_no = [1,2,3,4,5]
plt.figure()
plt.plot(epoch_no, epoch_loss, label="Loss against epoch")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

plt.xlim(left=0)
plt.ylim(bottom=0)
plt.show()

KeyboardInterrupt: 

### Step 8: Begin testing (**students are required to complete this part!**)

In [ ]:
# Load the fine-tuned model
ckpt_path = "../dpo/dpo.pt"
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
gpt = GPT(gptconf).cuda()
try:
    state_dict = checkpoint['model']
except:
    state_dict = checkpoint['model_state_dict']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
# Test
gpt.eval()
test_set = ["17+19=?", "3*17=?", "72/4=?", "72-x=34,x=?", "x*11=44,x=?", "3*17=?", "72/4=?", "72-x=34,x=?"]
with torch.no_grad():
    for prompt in test_set: 
        prompt_ids = encode(prompt)
        ###########################################################
        # Please complete the test code here!
        # ...
        # gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
        # ...
        ###########################################################